# 03 · Modelado Machine Learning

## ¿Para qué utilizo este notebook?

En este notebook agrego una capa de **Machine Learning** al proyecto de inteligencia financiera bancaria.

Hasta este punto ya tengo el dataset limpio, el análisis exploratorio y el dashboard descriptivo. Ahora voy a construir **3 modelos**:

| Modelo | Tipo | ¿Para qué lo utilizo? |
|---|---|---|
| `RandomForestRegressor` | Regresión supervisada | Para estimar el ROE y analizar qué variables ayudan a explicar la rentabilidad. |
| `KMeans` | Clustering no supervisado | Para segmentar bancos con perfiles financieros similares. |
| `Prophet` | Series temporales | Para proyectar tendencias futuras de los indicadores principales. |

## Archivos de entrada

```text
01_datos_procesados/dataset_financiero_limpio.parquet
```

o, si no existe:

```text
01_datos_procesados/dataset_financiero_limpio.csv
```

## Archivos de salida

Los resultados que usará el dashboard se guardan en:

```text
01_datos_procesados/modelos/
```

Los modelos serializados se guardan en:

```text
04_modelos/
```

## 1. Dependencias necesarias

### ¿Para qué utilizo este paso?

Aquí dejo documentadas las librerías necesarias para entrenar modelos.

Prophet suele ser sensible a versiones de `NumPy` y al backend `CmdStanPy`. Por eso dejo versiones fijas y una validación posterior dentro del notebook.

### Instalación recomendada

Ejecutar en la terminal, con el entorno virtual activado de conda python312:

```bash
conda install -c conda-forge prophet cmdstanpy cmdstan -y
```
Con esto tenemos instalado cmdStan

In [1]:
import sys

!{sys.executable} -m pip install scikit-learn==1.5.1 joblib==1.4.2 cmdstanpy==1.2.4 prophet==1.1.5 numpy==1.26.4

In [2]:
%pip install scikit-learn==1.5.1 joblib==1.4.2 cmdstanpy==1.2.4 prophet==1.1.5 numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Esta celda queda comentada para evitar reinstalar paquetes cada vez que ejecuto el notebook.
# Si mi entorno no tiene estas librerías, puedo descomentar y ejecutar.

# import sys
# !{sys.executable} -m pip install scikit-learn==1.5.1 joblib==1.4.2 cmdstanpy==1.2.4 prophet==1.1.5 numpy==1.26.4

import cmdstanpy
from prophet import Prophet

print("CmdStan path:", cmdstanpy.cmdstan_path())
print("Prophet importado correctamente")



c:\Users\eddy.trejo\AppData\Local\anaconda3\envs\python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


CmdStan path: C:\Users\eddy.trejo\AppData\Local\anaconda3\envs\python312\Library\bin\cmdstan
Prophet importado correctamente


## 2. Importar librerías y preparar compatibilidad

### ¿Para qué utilizo este paso?

Importo las librerías para transformar datos, entrenar modelos, evaluar resultados y guardar archivos.

También agrego una corrección defensiva para Prophet cuando el entorno usa NumPy 2.x y aparece el error de `np.float_`.

In [4]:
# pathlib permite manejar rutas del proyecto sin depender del sistema operativo.
from pathlib import Path

# warnings permite ocultar advertencias no críticas.
import warnings

# logging permite reducir mensajes técnicos de Prophet y CmdStanPy.
import logging

# sys permite conocer el Python que está usando el notebook.
import sys

# numpy se usa para cálculos numéricos y manejo de valores NaN.
import numpy as np

# pandas se usa para leer, transformar, agrupar y exportar datos.
import pandas as pd

# RandomForestRegressor se usa para estimar el ROE.
from sklearn.ensemble import RandomForestRegressor

# KMeans se usa para segmentar bancos.
from sklearn.cluster import KMeans

# PCA permite visualizar los clusters en dos dimensiones.
from sklearn.decomposition import PCA

# SimpleImputer rellena valores nulos usando la mediana.
from sklearn.impute import SimpleImputer

# Métricas para evaluar la regresión del ROE.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Pipeline encadena pasos: imputación + modelo.
from sklearn.pipeline import Pipeline

# StandardScaler estandariza variables para KMeans.
from sklearn.preprocessing import StandardScaler

# joblib guarda modelos entrenados en archivos .pkl.
import joblib

# os permite configurar variables del entorno de ejecución.
import os

# Indico a joblib/loky cuántos núcleos puede usar.
# Esto evita errores al intentar detectar automáticamente los núcleos físicos en Windows.
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

# Limito hilos internos usados por librerías numéricas.
# Esto ayuda a evitar advertencias o comportamientos raros con KMeans en Windows.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Oculto advertencias no críticas.
warnings.filterwarnings('ignore')

# Configuro visualización de tablas en pandas.
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 140)

# Reduzco logs extensos de Prophet/CmdStanPy.
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

# -------------------------------------------------------------------------
# Compatibilidad defensiva con NumPy 2.x
# -------------------------------------------------------------------------
# Algunas versiones de Prophet usan np.float_, atributo removido en NumPy 2.x.
# Si mi entorno usa NumPy 2.x, creo un alias temporal antes de importar Prophet.
if not hasattr(np, 'float_'):
    np.float_ = np.float64

# Intento importar Prophet de forma controlada.
try:
    from prophet import Prophet
    PROPHET_DISPONIBLE = True
    ERROR_PROPHET = None
except Exception as e:
    Prophet = None
    PROPHET_DISPONIBLE = False
    ERROR_PROPHET = repr(e)

# Muestro versiones clave para auditoría.
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Prophet disponible:', PROPHET_DISPONIBLE)

# Si Prophet no se pudo importar, muestro el detalle.
if not PROPHET_DISPONIBLE:
    print('Error Prophet:', ERROR_PROPHET)

Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]
NumPy: 2.4.3
Pandas: 3.0.3
Prophet disponible: True


## 3. Validar Prophet y CmdStanPy

### ¿Para qué utilizo este paso?

Prophet requiere un backend estadístico. En Python normalmente usa `CMDSTANPY`.

Este paso valida que el backend esté disponible. Si falta CmdStan, el notebook puede intentar instalarlo automáticamente.

> Si esta validación falla, los modelos RandomForest y KMeans siguen funcionando. Prophet se omite dejando registrado el motivo.

In [5]:
# Si está en True, el notebook intentará instalar CmdStan si no existe.
# Puede tardar varios minutos la primera vez.
INSTALAR_CMDSTAN_SI_FALTA = True

def validar_entorno_prophet(instalar_si_falta: bool = True):
    """Valida que Prophet y CmdStanPy estén listos para entrenar."""

    # Si Prophet no se pudo importar, no puedo entrenar proyecciones.
    if not PROPHET_DISPONIBLE:
        return False, f'Prophet no está disponible: {ERROR_PROPHET}'

    try:
        # Importo cmdstanpy para validar el backend.
        import cmdstanpy

        try:
            # Intento obtener la ruta de CmdStan.
            ruta_cmdstan = cmdstanpy.cmdstan_path()
            return True, f'CmdStan disponible en: {ruta_cmdstan}'

        except Exception as e_cmdstan:
            # Si falta CmdStan, intento instalarlo si la bandera está activada.
            if instalar_si_falta:
                print('CmdStan no está disponible. Intentando instalar CmdStan...')
                print('Este proceso puede tardar varios minutos.')
                cmdstanpy.install_cmdstan()
                ruta_cmdstan = cmdstanpy.cmdstan_path()
                return True, f'CmdStan instalado en: {ruta_cmdstan}'

            # Si no se instala automáticamente, devuelvo el motivo.
            return False, f'CmdStan no está disponible: {repr(e_cmdstan)}'

    except Exception as e:
        # Si cmdstanpy no está instalado o falla, devuelvo el motivo.
        return False, f'Error validando CmdStanPy: {repr(e)}'

# Ejecuto la validación de Prophet.
PROPHET_OK, MENSAJE_PROPHET = validar_entorno_prophet(INSTALAR_CMDSTAN_SI_FALTA)

# Muestro resultado de la validación.
print('Prophet listo para entrenar:', PROPHET_OK)
print(MENSAJE_PROPHET)

Prophet listo para entrenar: True
CmdStan disponible en: C:\Users\eddy.trejo\AppData\Local\anaconda3\envs\python312\Library\bin\cmdstan


## 4. Configurar rutas del proyecto

### ¿Para qué utilizo este paso?

Detecto la raíz del proyecto `GRUPO_02` y preparo rutas para leer datos y guardar resultados.

In [6]:
def encontrar_raiz_proyecto(nombre_carpeta_procesados: str = '01_datos_procesados') -> Path:
    """Busca la raíz del proyecto subiendo desde la carpeta actual."""

    # Obtengo carpeta actual.
    actual = Path.cwd().resolve()

    # Creo candidatos: carpeta actual y padres.
    candidatos = [actual] + list(actual.parents)

    # Recorro candidatos hasta encontrar 01_datos_procesados.
    for ruta in candidatos:
        if (ruta / nombre_carpeta_procesados).exists():
            return ruta

    # Si no se encuentra, detengo el proceso.
    raise FileNotFoundError(f'No se encontró {nombre_carpeta_procesados} desde {actual}')

# Raíz del proyecto.
RAIZ_PROYECTO = encontrar_raiz_proyecto()

# Carpeta de datos procesados.
CARPETA_PROCESADOS = RAIZ_PROYECTO / '01_datos_procesados'

# Carpeta donde se guardan resultados de modelos para dashboard.
CARPETA_RESULTADOS_MODELOS = CARPETA_PROCESADOS / 'modelos'

# Carpeta donde se guardan modelos serializados.
CARPETA_MODELOS = RAIZ_PROYECTO / '04_modelos'

# Crear carpetas si no existen.
CARPETA_RESULTADOS_MODELOS.mkdir(parents=True, exist_ok=True)
CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)

# Rutas del dataset limpio.
RUTA_DATASET_PARQUET = CARPETA_PROCESADOS / 'dataset_financiero_limpio.parquet'
RUTA_DATASET_CSV = CARPETA_PROCESADOS / 'dataset_financiero_limpio.csv'

# Muestro rutas para verificar.
print('Raíz:', RAIZ_PROYECTO)
print('Resultados modelos:', CARPETA_RESULTADOS_MODELOS)
print('Modelos serializados:', CARPETA_MODELOS)

Raíz: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador
Resultados modelos: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos
Modelos serializados: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\04_modelos


## 5. Definir parámetros generales

### ¿Para qué utilizo este paso?

Centralizo los indicadores y parámetros para no tener valores dispersos en el notebook.

In [7]:
# Indicadores esperados en el dataset.
INDICADORES_ESPERADOS = [
    'activos_totales',
    'pasivos_totales',
    'patrimonio',
    'roe',
    'morosidad',
    'solvencia_proxy'
]

# Indicadores que voy a proyectar con Prophet.
INDICADORES_PROPHET = [
    'activos_totales',
    'roe',
    'morosidad',
    'solvencia_proxy'
]

# Variables para estimar ROE.
FEATURES_REGRESION_ROE = [
    'activos_totales',
    'pasivos_totales',
    'patrimonio',
    'morosidad',
    'solvencia_proxy'
]

# Variable objetivo de la regresión.
TARGET_ROE = 'roe'

# Variables para clustering.
FEATURES_CLUSTER = [
    'activos_totales',
    'patrimonio',
    'roe',
    'morosidad',
    'solvencia_proxy'
]

# Número de clusters financieros.
N_CLUSTERS = 4

# Horizonte de proyección en meses.
HORIZONTE_PROPHET_MESES = 12

# Mínimo de observaciones para entrenar Prophet.
MIN_OBSERVACIONES_PROPHET = 24

# Para pruebas rápidas puedo poner 20; para todas las series dejo None.
MAX_SERIES_PROPHET = None

# Semilla para reproducibilidad.
RANDOM_STATE = 42

## 6. Cargar y preparar dataset

### ¿Para qué utilizo este paso?

Cargo el dataset limpio, normalizo columnas y creo variables de fecha necesarias para los modelos.

In [8]:
def cargar_dataset_financiero() -> pd.DataFrame:
    """Carga el dataset limpio desde Parquet o CSV."""

    # Prioridad 1: Parquet.
    if RUTA_DATASET_PARQUET.exists():
        print('Leyendo Parquet:', RUTA_DATASET_PARQUET)
        return pd.read_parquet(RUTA_DATASET_PARQUET)

    # Prioridad 2: CSV.
    if RUTA_DATASET_CSV.exists():
        print('Leyendo CSV:', RUTA_DATASET_CSV)
        return pd.read_csv(RUTA_DATASET_CSV)

    # Si no existe ninguno, detengo.
    raise FileNotFoundError('No se encontró dataset_financiero_limpio')

# Cargo dataset largo.
df_largo = cargar_dataset_financiero().copy()

# Limpio periodo.
df_largo['periodo'] = df_largo['periodo'].astype(str).str.strip()

# Limpio banco.
df_largo['banco_estandarizado'] = df_largo['banco_estandarizado'].astype(str).str.strip().str.upper()

# Limpio indicador.
df_largo['indicador'] = df_largo['indicador'].astype(str).str.strip().str.lower()

# Limpio unidad.
df_largo['unidad'] = df_largo['unidad'].astype(str).str.strip().str.lower()

# Limpio sentido.
df_largo['sentido'] = df_largo['sentido'].astype(str).str.strip().str.lower()

# Aseguro que valor sea numérico.
df_largo['valor'] = pd.to_numeric(df_largo['valor'], errors='coerce')

# Convierto periodo YYYY-MM a fecha mensual.
df_largo['periodo_dt'] = pd.to_datetime(df_largo['periodo'] + '-01', errors='coerce')

# Extraigo año.
df_largo['anio'] = df_largo['periodo_dt'].dt.year

# Extraigo mes.
df_largo['mes'] = df_largo['periodo_dt'].dt.month

# Filtro indicadores esperados.
df_largo = df_largo[df_largo['indicador'].isin(INDICADORES_ESPERADOS)].copy()

# Muestro resumen.
print('Filas:', len(df_largo))
print('Bancos:', df_largo['banco_estandarizado'].nunique())
print('Periodos:', df_largo['periodo'].nunique())
print('Indicadores:', sorted(df_largo['indicador'].unique()))

display(df_largo.head(10))

Leyendo Parquet: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_financiero_limpio.parquet
Filas: 29730
Bancos: 32
Periodos: 205
Indicadores: ['activos_totales', 'morosidad', 'pasivos_totales', 'patrimonio', 'roe', 'solvencia_proxy']


,periodo,banco_estandarizado,indicador,valor,unidad,sentido,periodo_dt,anio,mes
0,2009-01,AMAZONAS,activos_totales,111.57,millones_usd,mayor_es_tamano,2009-01-01,2009,1
1,2009-01,AMAZONAS,morosidad,4.55,porcentaje,menor_es_mejor,2009-01-01,2009,1
2,2009-01,AMAZONAS,pasivos_totales,100.08,millones_usd,informativo,2009-01-01,2009,1
3,2009-01,AMAZONAS,patrimonio,11.38,millones_usd,mayor_es_mejor,2009-01-01,2009,1
4,2009-01,AMAZONAS,roe,11.41,porcentaje,mayor_es_mejor,2009-01-01,2009,1
5,2009-01,AMAZONAS,solvencia_proxy,8.65,porcentaje,mayor_es_mejor,2009-01-01,2009,1
6,2009-01,AUSTRO,activos_totales,646.02,millones_usd,mayor_es_tamano,2009-01-01,2009,1
7,2009-01,AUSTRO,morosidad,6.10,porcentaje,menor_es_mejor,2009-01-01,2009,1
8,2009-01,AUSTRO,pasivos_totales,588.88,millones_usd,informativo,2009-01-01,2009,1
9,2009-01,AUSTRO,patrimonio,56.66,millones_usd,mayor_es_mejor,2009-01-01,2009,1


## 7. Convertir a formato ancho y crear dataset anual

### ¿Para qué utilizo este paso?

Los modelos necesitan una columna por indicador. También creo un dataset anual para clustering.

In [9]:
# Transformo de formato largo a formato ancho.
df_wide = (
    df_largo
    .pivot_table(
        index=['periodo', 'periodo_dt', 'anio', 'mes', 'banco_estandarizado'],
        columns='indicador',
        values='valor',
        aggfunc='first'
    )
    .reset_index()
)

# Aseguro todas las columnas de indicadores.
for indicador in INDICADORES_ESPERADOS:
    if indicador not in df_wide.columns:
        df_wide[indicador] = np.nan

# Ordeno columnas.
df_wide = df_wide[['periodo', 'periodo_dt', 'anio', 'mes', 'banco_estandarizado'] + INDICADORES_ESPERADOS].copy()

# Ordeno filas.
df_wide = df_wide.sort_values(['periodo_dt', 'banco_estandarizado']).reset_index(drop=True)

# Construyo dataset anual promedio por banco.
df_anual = (
    df_wide
    .groupby(['anio', 'banco_estandarizado'], dropna=False)
    .agg(
        activos_totales=('activos_totales', 'mean'),
        pasivos_totales=('pasivos_totales', 'mean'),
        patrimonio=('patrimonio', 'mean'),
        roe=('roe', 'mean'),
        morosidad=('morosidad', 'mean'),
        solvencia_proxy=('solvencia_proxy', 'mean'),
        meses_con_dato=('periodo', 'nunique')
    )
    .reset_index()
)

# Muestro controles.
print('Dataset mensual ancho:', df_wide.shape)
print('Dataset anual:', df_anual.shape)

display(df_wide.head(5))
display(df_anual.head(5))

Dataset mensual ancho: (4958, 11)
Dataset anual: (441, 9)


indicador,periodo,periodo_dt,anio,mes,banco_estandarizado,activos_totales,pasivos_totales,patrimonio,roe,morosidad,solvencia_proxy
0,2009-01,2009-01-01,2009,1,AMAZONAS,111.57,100.08,11.38,11.41,4.55,8.65
1,2009-01,2009-01-01,2009,1,AUSTRO,646.02,588.88,56.66,10.11,6.10,7.52
2,2009-01,2009-01-01,2009,1,BOLIVARIANO,1271.10,1157.19,112.64,13.53,1.39,7.84
3,2009-01,2009-01-01,2009,1,CAPITAL,65.80,54.06,11.72,2.82,10.98,15.37
4,2009-01,2009-01-01,2009,1,CITIBANK,319.13,289.80,28.96,15.54,4.22,7.70


,anio,banco_estandarizado,activos_totales,pasivos_totales,patrimonio,roe,morosidad,solvencia_proxy,meses_con_dato
0,2009,AMAZONAS,120.492500,108.550833,11.405833,8.990000,3.854167,7.836667,12
1,2009,AUSTRO,647.960833,585.148333,57.102500,20.646667,7.670000,7.409167,12
2,2009,BOLIVARIANO,1271.975833,1155.695833,109.273333,13.958333,1.559167,7.487500,12
3,2009,CAPITAL,62.264167,49.837500,11.842500,10.010833,10.998333,16.355000,12
4,2009,CITIBANK,302.596667,271.880000,29.135833,13.300000,16.335833,8.315000,12


## 8. Función de exportación

### ¿Para qué utilizo este paso?

Exporto cada resultado en CSV y Parquet para que el dashboard pueda leer rápido y también se pueda revisar manualmente.

In [10]:
def exportar_csv_parquet(df_exportar: pd.DataFrame, nombre_base: str):
    """Exporta un DataFrame a CSV y Parquet."""

    # Ruta CSV.
    ruta_csv = CARPETA_RESULTADOS_MODELOS / f'{nombre_base}.csv'

    # Ruta Parquet.
    ruta_parquet = CARPETA_RESULTADOS_MODELOS / f'{nombre_base}.parquet'

    # Exporto CSV.
    df_exportar.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

    # Exporto Parquet si el entorno lo permite.
    try:
        df_exportar.to_parquet(ruta_parquet, index=False)
        print('Exportado:', ruta_csv)
        print('Exportado:', ruta_parquet)
    except Exception as e:
        print('Exportado CSV:', ruta_csv)
        print('No se pudo exportar Parquet:', repr(e))

# Modelo 1 · RandomForestRegressor para ROE

## ¿Para qué utilizo este modelo?

Uso este modelo para estimar el **ROE** a partir de otros indicadores financieros.

Esto me permite analizar si variables como morosidad, patrimonio, activos y solvencia proxy ayudan a explicar la rentabilidad.

In [11]:
# Selecciono filas con ROE disponible.
df_regresion = df_wide.dropna(subset=[TARGET_ROE]).copy()

# Conservo filas con al menos una variable predictora con dato.
df_regresion = df_regresion[df_regresion[FEATURES_REGRESION_ROE].notna().any(axis=1)].copy()

# Valido cantidad mínima de datos.
if len(df_regresion) < 50:
    raise ValueError('No hay suficientes datos para entrenar el modelo ROE.')

# Ordeno periodos para dividir entrenamiento/prueba temporalmente.
periodos_ordenados = sorted(df_regresion['periodo'].dropna().unique())

# Punto de corte 80% train, 20% test.
punto_corte = max(1, int(len(periodos_ordenados) * 0.80))

# Periodos de entrenamiento y prueba.
periodos_train = periodos_ordenados[:punto_corte]
periodos_test = periodos_ordenados[punto_corte:]

# Dataset train.
df_train = df_regresion[df_regresion['periodo'].isin(periodos_train)].copy()

# Dataset test.
df_test = df_regresion[df_regresion['periodo'].isin(periodos_test)].copy()

# Variables predictoras y objetivo.
X_train = df_train[FEATURES_REGRESION_ROE]
y_train = df_train[TARGET_ROE]
X_test = df_test[FEATURES_REGRESION_ROE]
y_test = df_test[TARGET_ROE]

# Pipeline: imputar nulos + Random Forest.
modelo_roe_eval = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Entreno el modelo de evaluación.
modelo_roe_eval.fit(X_train, y_train)

# Predicciones train y test.
pred_train = modelo_roe_eval.predict(X_train)
pred_test = modelo_roe_eval.predict(X_test) if len(df_test) > 0 else np.array([])

# Métricas del modelo.
metricas = [{
    'modelo': 'RandomForestRegressor_ROE',
    'conjunto': 'train',
    'mae': mean_absolute_error(y_train, pred_train),
    'rmse': np.sqrt(mean_squared_error(y_train, pred_train)),
    'r2': r2_score(y_train, pred_train),
    'observaciones': len(y_train)
}]

# Métricas test si existe.
if len(df_test) > 0:
    metricas.append({
        'modelo': 'RandomForestRegressor_ROE',
        'conjunto': 'test',
        'mae': mean_absolute_error(y_test, pred_test),
        'rmse': np.sqrt(mean_squared_error(y_test, pred_test)),
        'r2': r2_score(y_test, pred_test),
        'observaciones': len(y_test)
    })

# DataFrame de métricas.
df_metricas_roe = pd.DataFrame(metricas)

# Modelo final con todos los datos disponibles.
modelo_roe_final = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Entreno modelo final.
X_all = df_regresion[FEATURES_REGRESION_ROE]
y_all = df_regresion[TARGET_ROE]
modelo_roe_final.fit(X_all, y_all)

# Dataset de predicciones.
df_pred_roe = df_regresion[['periodo', 'periodo_dt', 'anio', 'mes', 'banco_estandarizado'] + FEATURES_REGRESION_ROE + [TARGET_ROE]].copy()

# ROE real.
df_pred_roe['roe_real'] = df_pred_roe[TARGET_ROE]

# ROE estimado.
df_pred_roe['roe_estimado'] = modelo_roe_final.predict(X_all)

# Error.
df_pred_roe['error_roe'] = df_pred_roe['roe_real'] - df_pred_roe['roe_estimado']

# Error absoluto.
df_pred_roe['error_abs_roe'] = df_pred_roe['error_roe'].abs()

# Importancia de variables.
df_importancia_roe = pd.DataFrame({
    'variable': FEATURES_REGRESION_ROE,
    'importancia': modelo_roe_final.named_steps['model'].feature_importances_
}).sort_values('importancia', ascending=False)

# Guardo modelo final.
joblib.dump(modelo_roe_final, CARPETA_MODELOS / 'modelo_roe_random_forest.pkl')

# Exporto resultados.
exportar_csv_parquet(df_pred_roe, 'modelo_roe_predicciones')
exportar_csv_parquet(df_metricas_roe, 'modelo_roe_metricas')
exportar_csv_parquet(df_importancia_roe, 'modelo_roe_importancia_variables')

# Muestro resultados.
display(df_metricas_roe)
display(df_importancia_roe)
display(df_pred_roe.head(10))

Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_predicciones.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_predicciones.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_metricas.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_metricas.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_importancia_variables.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_importancia_variables.pa

,modelo,conjunto,mae,rmse,r2,observaciones
0,RandomForestRegressor_ROE,train,3.974079,9.044939,0.743238,3982
1,RandomForestRegressor_ROE,test,11.607169,20.372345,0.059112,949


,variable,importancia
2,patrimonio,0.266480
3,morosidad,0.241312
4,solvencia_proxy,0.178108
0,activos_totales,0.157080
1,pasivos_totales,0.157020


indicador,periodo,periodo_dt,anio,mes,banco_estandarizado,activos_totales,pasivos_totales,patrimonio,morosidad,solvencia_proxy,roe,roe_real,roe_estimado,error_roe,error_abs_roe
0,2009-01,2009-01-01,2009,1,AMAZONAS,111.57,100.08,11.38,4.55,8.65,11.41,11.41,8.223050,3.186950,3.186950
1,2009-01,2009-01-01,2009,1,AUSTRO,646.02,588.88,56.66,6.10,7.52,10.11,10.11,12.606557,-2.496557,2.496557
2,2009-01,2009-01-01,2009,1,BOLIVARIANO,1271.10,1157.19,112.64,1.39,7.84,13.53,13.53,13.107103,0.422897,0.422897
3,2009-01,2009-01-01,2009,1,CAPITAL,65.80,54.06,11.72,10.98,15.37,2.82,2.82,10.784801,-7.964801,7.964801
4,2009-01,2009-01-01,2009,1,CITIBANK,319.13,289.80,28.96,4.22,7.70,15.54,15.54,17.725402,-2.185402,2.185402
5,2009-01,2009-01-01,2009,1,COFIEC,12.37,3.98,8.42,23.99,53.02,-3.89,-3.89,9.764918,-13.654918,13.654918
6,2009-01,2009-01-01,2009,1,COMERCIAL DE MANABI,33.80,27.52,6.28,3.86,13.22,43.95,43.95,22.700730,21.249270,21.249270
7,2009-01,2009-01-01,2009,1,DELBANK,18.09,10.74,7.34,3.63,28.54,32.09,32.09,26.304019,5.785981,5.785981
8,2009-01,2009-01-01,2009,1,FINCA,37.77,27.51,10.21,10.56,24.22,5.97,5.97,5.387295,0.582705,0.582705
9,2009-01,2009-01-01,2009,1,GENERAL RUMIÑAHUI,324.85,303.03,21.50,3.60,5.83,18.03,18.03,20.650172,-2.620172,2.620172


# Modelo 2 · KMeans para segmentación bancaria

## ¿Para qué utilizo este modelo?

Uso `KMeans` para agrupar bancos con perfiles financieros similares según sus promedios anuales.

Este modelo ayuda a identificar grupos como bancos grandes, bancos intermedios, bancos pequeños con alta solvencia proxy o bancos con mayor riesgo relativo.

In [12]:
# Copio dataset anual para clustering.
df_cluster = df_anual.copy()

# Mantengo filas con al menos una variable útil.
df_cluster = df_cluster[df_cluster[FEATURES_CLUSTER].notna().any(axis=1)].copy()

# Valido observaciones suficientes.
if len(df_cluster) < N_CLUSTERS:
    raise ValueError('No hay suficientes observaciones para KMeans.')

# Matriz de variables para clustering.
X_cluster = df_cluster[FEATURES_CLUSTER]

# Imputo nulos con mediana.
imputer_cluster = SimpleImputer(strategy='median')
X_cluster_imputado = imputer_cluster.fit_transform(X_cluster)

# Escalo variables para que activos no domine el clustering.
scaler_cluster = StandardScaler()
X_cluster_escalado = scaler_cluster.fit_transform(X_cluster_imputado)

# Creo modelo KMeans.
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)

# Entreno y asigno cluster.
clusters = kmeans.fit_predict(X_cluster_escalado)

# PCA para coordenadas 2D de visualización.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords_pca = pca.fit_transform(X_cluster_escalado)

# Dataset final de clusters.
df_clusters = df_cluster[['anio', 'banco_estandarizado', 'meses_con_dato'] + FEATURES_CLUSTER].copy()
df_clusters['cluster'] = clusters
df_clusters['pca_1'] = coords_pca[:, 0]
df_clusters['pca_2'] = coords_pca[:, 1]

# Resumen por cluster.
resumen_cluster = (
    df_clusters
    .groupby('cluster')
    .agg(
        bancos=('banco_estandarizado', 'nunique'),
        observaciones=('banco_estandarizado', 'size'),
        activos_promedio=('activos_totales', 'mean'),
        patrimonio_promedio=('patrimonio', 'mean'),
        roe_promedio=('roe', 'mean'),
        morosidad_promedio=('morosidad', 'mean'),
        solvencia_promedio=('solvencia_proxy', 'mean')
    )
    .reset_index()
)

# Medianas globales para interpretar clusters.
mediana_activos = df_clusters['activos_totales'].median()
mediana_morosidad = df_clusters['morosidad'].median()
mediana_solvencia = df_clusters['solvencia_proxy'].median()
mediana_roe = df_clusters['roe'].median()

# Función simple de interpretación de clusters.
def asignar_perfil_cluster(fila):
    if fila['activos_promedio'] >= mediana_activos and fila['morosidad_promedio'] <= mediana_morosidad:
        return 'Bancos grandes o consolidados'
    if fila['morosidad_promedio'] > mediana_morosidad and fila['roe_promedio'] < mediana_roe:
        return 'Mayor riesgo relativo'
    if fila['activos_promedio'] < mediana_activos and fila['solvencia_promedio'] >= mediana_solvencia:
        return 'Pequeños con mayor solvencia proxy'
    return 'Perfil intermedio'

# Aplico interpretación.
resumen_cluster['perfil_cluster'] = resumen_cluster.apply(asignar_perfil_cluster, axis=1)

# Agrego perfil al detalle.
df_clusters = df_clusters.merge(resumen_cluster[['cluster', 'perfil_cluster']], on='cluster', how='left')

# Guardo modelo y transformadores.
joblib.dump({
    'imputer': imputer_cluster,
    'scaler': scaler_cluster,
    'kmeans': kmeans,
    'pca': pca,
    'features': FEATURES_CLUSTER
}, CARPETA_MODELOS / 'modelo_clusters_kmeans.pkl')

# Exporto resultados.
exportar_csv_parquet(df_clusters, 'clusters_bancos_anual')
exportar_csv_parquet(resumen_cluster, 'clusters_resumen')

# Muestro resultados.
display(resumen_cluster)
display(df_clusters.head(10))

Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\clusters_bancos_anual.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\clusters_bancos_anual.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\clusters_resumen.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\clusters_resumen.parquet


,cluster,bancos,observaciones,activos_promedio,patrimonio_promedio,roe_promedio,morosidad_promedio,solvencia_promedio,perfil_cluster
0,0,6,53,8230.340425,794.860600,12.370240,3.641634,8.673702,Bancos grandes o consolidados
1,1,10,58,90.881565,19.271540,22.666224,6.469026,26.359243,Pequeños con mayor solvencia proxy
2,2,29,311,891.734290,94.833309,9.424884,6.126650,10.703086,Mayor riesgo relativo
3,3,7,19,1689.129097,152.368829,4.101456,64.333434,16.176182,Mayor riesgo relativo


,anio,banco_estandarizado,meses_con_dato,activos_totales,patrimonio,roe,morosidad,solvencia_proxy,cluster,pca_1,pca_2,perfil_cluster
0,2009,AMAZONAS,12,120.492500,11.405833,8.990000,3.854167,7.836667,2,-0.475294,-0.480742,Mayor riesgo relativo
1,2009,AUSTRO,12,647.960833,57.102500,20.646667,7.670000,7.409167,2,-0.223535,-0.793598,Mayor riesgo relativo
2,2009,BOLIVARIANO,12,1271.975833,109.273333,13.958333,1.559167,7.487500,2,0.071318,-0.718681,Mayor riesgo relativo
3,2009,CAPITAL,12,62.264167,11.842500,10.010833,10.998333,16.355000,2,-0.916158,0.123199,Mayor riesgo relativo
4,2009,CITIBANK,12,302.596667,29.135833,13.300000,16.335833,8.315000,2,-0.538688,-0.005709,Mayor riesgo relativo
5,2009,COFIEC,12,12.694167,8.422500,16.884167,13.357500,49.340000,1,-2.285805,1.028695,Pequeños con mayor solvencia proxy
6,2009,COMERCIAL DE MANABI,12,31.005833,6.552500,9.370000,6.274167,15.397500,2,-0.843769,-0.126826,Mayor riesgo relativo
7,2009,DELBANK,12,16.391667,7.345833,20.746667,4.370833,34.221667,1,-1.542175,-0.128875,Pequeños con mayor solvencia proxy
8,2009,FINCA,12,37.073333,10.245000,4.708333,12.186667,24.007500,1,-1.279572,0.692629,Pequeños con mayor solvencia proxy
9,2009,GENERAL RUMIÑAHUI,12,340.760833,21.722500,18.901667,4.043333,5.357500,2,-0.254812,-1.006472,Mayor riesgo relativo


# Modelo 3 · Prophet para proyecciones

## ¿Para qué utilizo este modelo?

Uso Prophet para generar proyecciones exploratorias por banco e indicador.

Cada serie tiene esta forma:

```text
banco_estandarizado + indicador
```

Ejemplo:

```text
PICHINCHA + activos_totales
AUSTRO + morosidad
GUAYAQUIL + roe
```

Estas proyecciones no son una recomendación financiera; son una estimación estadística basada en tendencia histórica.

In [13]:
def entrenar_prophet_serie(df_serie: pd.DataFrame, banco: str, indicador: str, horizonte_meses: int):
    """Entrena Prophet para una serie banco-indicador."""

    # Elimino nulos y ordeno por fecha.
    serie = df_serie.dropna(subset=['valor']).sort_values('periodo_dt').copy()

    # Si hay pocas observaciones, omito la serie.
    if len(serie) < MIN_OBSERVACIONES_PROPHET:
        return None, {
            'banco_estandarizado': banco,
            'indicador': indicador,
            'estado': 'omitido',
            'motivo': f'menos de {MIN_OBSERVACIONES_PROPHET} observaciones',
            'observaciones': len(serie),
            'mae_ajuste': np.nan,
            'rmse_ajuste': np.nan
        }

    # Prophet requiere columnas ds y y.
    df_prophet = serie[['periodo_dt', 'valor']].rename(columns={'periodo_dt': 'ds', 'valor': 'y'})

    # Creo modelo Prophet usando backend CMDSTANPY.
    modelo = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.80,
        stan_backend='CMDSTANPY'
    )

    # Entreno el modelo.
    modelo.fit(df_prophet)

    # Creo fechas futuras mensuales.
    futuro = modelo.make_future_dataframe(periods=horizonte_meses, freq='MS', include_history=True)

    # Genero forecast.
    forecast = modelo.predict(futuro)

    # Selecciono columnas importantes.
    resultado = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend']].copy()

    # Agrego valor real histórico.
    resultado = resultado.merge(df_prophet.rename(columns={'y': 'valor_real'}), on='ds', how='left')

    # Identifico última fecha histórica.
    ultimo_ds_historico = df_prophet['ds'].max()

    # Marco si es histórico ajustado o proyección futura.
    resultado['tipo'] = np.where(resultado['ds'] <= ultimo_ds_historico, 'historico_ajustado', 'proyeccion')

    # Creo columnas auxiliares.
    resultado['periodo'] = resultado['ds'].dt.strftime('%Y-%m')
    resultado['anio'] = resultado['ds'].dt.year
    resultado['mes'] = resultado['ds'].dt.month
    resultado['banco_estandarizado'] = banco
    resultado['indicador'] = indicador

    # Evalúo ajuste sobre los puntos históricos.
    hist_eval = resultado[resultado['valor_real'].notna()].copy()
    mae = mean_absolute_error(hist_eval['valor_real'], hist_eval['yhat'])
    rmse = np.sqrt(mean_squared_error(hist_eval['valor_real'], hist_eval['yhat']))

    # Registro métricas.
    metrica = {
        'banco_estandarizado': banco,
        'indicador': indicador,
        'estado': 'entrenado',
        'motivo': None,
        'observaciones': len(serie),
        'mae_ajuste': mae,
        'rmse_ajuste': rmse
    }

    return resultado, metrica

## Ejecutar Prophet y exportar proyecciones

### ¿Para qué utilizo este paso?

Recorro todas las series candidatas y genero proyecciones. Si una serie falla, registro el error y continúo con las demás.

In [14]:
# Si Prophet no está listo, creo archivos vacíos controlados.
if not PROPHET_OK:
    print('Prophet no está listo para entrenar.')
    print(MENSAJE_PROPHET)

    # Estructura vacía de forecast.
    df_prophet_forecast = pd.DataFrame(columns=[
        'ds', 'periodo', 'anio', 'mes', 'banco_estandarizado', 'indicador',
        'valor_real', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'tipo'
    ])

    # Métrica con motivo de no ejecución.
    df_prophet_metricas = pd.DataFrame([{
        'banco_estandarizado': None,
        'indicador': None,
        'estado': 'no_ejecutado',
        'motivo': MENSAJE_PROPHET,
        'observaciones': 0,
        'mae_ajuste': np.nan,
        'rmse_ajuste': np.nan
    }])

else:
    # Lista para guardar forecasts.
    resultados_forecast = []

    # Lista para guardar métricas.
    metricas_prophet = []

    # Contador de series entrenadas.
    contador_series = 0

    # Series candidatas banco + indicador.
    series_disponibles = (
        df_largo[df_largo['indicador'].isin(INDICADORES_PROPHET)]
        .groupby(['banco_estandarizado', 'indicador'])
        .size()
        .reset_index(name='observaciones')
        .sort_values(['indicador', 'banco_estandarizado'])
    )

    print('Series candidatas para Prophet:', len(series_disponibles))

    # Recorro cada serie.
    for _, fila in series_disponibles.iterrows():
        banco = fila['banco_estandarizado']
        indicador = fila['indicador']

        # Si estoy haciendo prueba rápida, respeto el límite.
        if MAX_SERIES_PROPHET is not None and contador_series >= MAX_SERIES_PROPHET:
            break

        # Extraigo serie específica.
        df_serie = df_largo[
            (df_largo['banco_estandarizado'] == banco)
            & (df_largo['indicador'] == indicador)
        ][['periodo', 'periodo_dt', 'valor']].copy()

        try:
            # Entreno Prophet para la serie.
            forecast, metrica = entrenar_prophet_serie(df_serie, banco, indicador, HORIZONTE_PROPHET_MESES)

            # Guardo métrica.
            metricas_prophet.append(metrica)

            # Guardo forecast si existe.
            if forecast is not None:
                resultados_forecast.append(forecast)
                contador_series += 1

            # Mensaje de avance.
            if contador_series % 10 == 0 and contador_series > 0:
                print(f'Series Prophet entrenadas: {contador_series}')

        except Exception as e:
            # Registro error de la serie y continúo.
            metricas_prophet.append({
                'banco_estandarizado': banco,
                'indicador': indicador,
                'estado': 'error',
                'motivo': repr(e),
                'observaciones': len(df_serie),
                'mae_ajuste': np.nan,
                'rmse_ajuste': np.nan
            })

    # Uno forecasts exitosos.
    if resultados_forecast:
        df_prophet_forecast = pd.concat(resultados_forecast, ignore_index=True)
    else:
        df_prophet_forecast = pd.DataFrame(columns=[
            'ds', 'periodo', 'anio', 'mes', 'banco_estandarizado', 'indicador',
            'valor_real', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'tipo'
        ])

    # Convierto métricas a DataFrame.
    df_prophet_metricas = pd.DataFrame(metricas_prophet)

# Exporto resultados Prophet.
exportar_csv_parquet(df_prophet_forecast, 'prophet_forecast')
exportar_csv_parquet(df_prophet_metricas, 'prophet_metricas_ajuste')

# Muestro resumen.
print('Filas forecast Prophet:', len(df_prophet_forecast))
display(df_prophet_metricas.head(20))
display(df_prophet_forecast.head(10))

20:16:22 - cmdstanpy - INFO - Chain [1] start processing


Series candidatas para Prophet: 128


20:16:23 - cmdstanpy - INFO - Chain [1] done processing
20:16:23 - cmdstanpy - INFO - Chain [1] start processing
20:16:23 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:24 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:24 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:24 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:24 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:24 - cmdstanpy - INFO - Chain [1] done processing
20:16:24 - cmdstanpy - INFO - Chain [1] start processing
20:16:25 - cmdstanpy - INFO - Chain [1] done processing
20:16:25 - cmdstanpy - INFO - Chain [1] start processing
20:16:25 - cmdstanpy - INFO - Chain [1] done processing
20:16:25 - cmdstanpy - INFO - Chain [1] 

Series Prophet entrenadas: 10


20:16:25 - cmdstanpy - INFO - Chain [1] start processing
20:16:25 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:26 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:26 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:26 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:26 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:26 - cmdstanpy - INFO - Chain [1] done processing
20:16:26 - cmdstanpy - INFO - Chain [1] start processing
20:16:27 - cmdstanpy - INFO - Chain [1] done processing
20:16:27 - cmdstanpy - INFO - Chain [1] start processing
20:16:27 - cmdstanpy - INFO - Chain [1] done processing
20:16:27 - cmdstanpy - INFO - Chain [1] start processing
20:16:27 - cmdstanpy - INFO - Chain [1]

...

Series Prophet entrenadas: 110
Series Prophet entrenadas: 110


20:16:43 - cmdstanpy - INFO - Chain [1] start processing
20:16:43 - cmdstanpy - INFO - Chain [1] done processing
20:16:43 - cmdstanpy - INFO - Chain [1] start processing
20:16:43 - cmdstanpy - INFO - Chain [1] done processing
20:16:43 - cmdstanpy - INFO - Chain [1] start processing
20:16:43 - cmdstanpy - INFO - Chain [1] done processing
20:16:43 - cmdstanpy - INFO - Chain [1] start processing
20:16:43 - cmdstanpy - INFO - Chain [1] done processing
20:16:43 - cmdstanpy - INFO - Chain [1] start processing
20:16:44 - cmdstanpy - INFO - Chain [1] done processing
20:16:44 - cmdstanpy - INFO - Chain [1] start processing
20:16:44 - cmdstanpy - INFO - Chain [1] done processing
20:16:44 - cmdstanpy - INFO - Chain [1] start processing
20:16:44 - cmdstanpy - INFO - Chain [1] done processing
20:16:44 - cmdstanpy - INFO - Chain [1] start processing
20:16:44 - cmdstanpy - INFO - Chain [1] done processing
20:16:45 - cmdstanpy - INFO - Chain [1] start processing
20:16:45 - cmdstanpy - INFO - Chain [1]

Series Prophet entrenadas: 120
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\prophet_forecast.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\prophet_forecast.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\prophet_metricas_ajuste.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\prophet_metricas_ajuste.parquet
Filas forecast Prophet: 21056


,banco_estandarizado,indicador,estado,motivo,observaciones,mae_ajuste,rmse_ajuste
0,AMAZONAS,activos_totales,entrenado,NaN,205,7.061681,9.637856
1,ATLANTIDA,activos_totales,omitido,menos de 24 observaciones,10,NaN,NaN
2,AUSTRO,activos_totales,entrenado,NaN,205,67.506239,92.811241
3,BOLIVARIANO,activos_totales,entrenado,NaN,205,59.786668,82.876901
4,CAPITAL,activos_totales,entrenado,NaN,205,29.310577,34.702275
5,CITIBANK,activos_totales,entrenado,NaN,205,36.554724,46.280842
6,COFIEC,activos_totales,entrenado,NaN,79,7.180468,9.080764
7,COMERCIAL DE MANABI,activos_totales,entrenado,NaN,205,4.428824,6.576856
8,COOPNACIONAL,activos_totales,entrenado,NaN,173,3.035398,4.005093
9,D-MIRO,activos_totales,entrenado,NaN,164,6.997392,8.682546


,ds,yhat,yhat_lower,yhat_upper,trend,valor_real,tipo,periodo,anio,mes,banco_estandarizado,indicador
0,2009-01-01,118.374275,105.267850,130.967739,116.416514,111.57,historico_ajustado,2009-01,2009,1,AMAZONAS,activos_totales
1,2009-02-01,123.488488,110.729991,135.996795,116.966778,113.85,historico_ajustado,2009-02,2009,2,AMAZONAS,activos_totales
2,2009-03-01,119.103193,106.472930,131.271296,117.463790,114.34,historico_ajustado,2009-03,2009,3,AMAZONAS,activos_totales
3,2009-04-01,117.861102,105.754325,129.802262,118.014054,115.09,historico_ajustado,2009-04,2009,4,AMAZONAS,activos_totales
4,2009-05-01,120.744474,107.626556,133.218136,118.546568,115.35,historico_ajustado,2009-05,2009,5,AMAZONAS,activos_totales
5,2009-06-01,120.058387,107.447968,132.575842,119.096832,119.15,historico_ajustado,2009-06,2009,6,AMAZONAS,activos_totales
6,2009-07-01,119.542739,107.432299,132.462263,119.629345,123.79,historico_ajustado,2009-07,2009,7,AMAZONAS,activos_totales
7,2009-08-01,121.229396,109.653165,133.986935,120.179609,127.19,historico_ajustado,2009-08,2009,8,AMAZONAS,activos_totales
8,2009-09-01,120.793892,108.633881,133.095026,120.729873,127.86,historico_ajustado,2009-09,2009,9,AMAZONAS,activos_totales
9,2009-10-01,121.985132,109.803609,134.538775,121.262387,131.89,historico_ajustado,2009-10,2009,10,AMAZONAS,activos_totales


# Resumen final

## ¿Qué generé?

Este notebook genera resultados para 3 modelos:

| Modelo | Archivos principales |
|---|---|
| RandomForestRegressor | `modelo_roe_predicciones`, `modelo_roe_metricas`, `modelo_roe_importancia_variables` |
| KMeans | `clusters_bancos_anual`, `clusters_resumen` |
| Prophet | `prophet_forecast`, `prophet_metricas_ajuste` |

## ¿Dónde se guardan?

```text
01_datos_procesados/modelos/
```

## ¿Qué sigue?

Actualizar `app_dashboard_v2.py` para agregar una pestaña:

```text
Modelos ML
```

con secciones para:

```text
Regresión ROE | Segmentación KMeans | Proyecciones Prophet
```

In [15]:
# Muestro archivos generados en la carpeta de modelos procesados.
print('Archivos generados en:', CARPETA_RESULTADOS_MODELOS)
for ruta in sorted(CARPETA_RESULTADOS_MODELOS.glob('*')):
    print('-', ruta.name)

# Muestro modelos serializados.
print('Modelos serializados en:', CARPETA_MODELOS)
for ruta in sorted(CARPETA_MODELOS.glob('*')):
    print('-', ruta.name)

Archivos generados en: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos
- clusters_bancos_anual.csv
- clusters_bancos_anual.parquet
- clusters_resumen.csv
- clusters_resumen.parquet
- modelo_roe_importancia_variables.csv
- modelo_roe_importancia_variables.parquet
- modelo_roe_metricas.csv
- modelo_roe_metricas.parquet
- modelo_roe_predicciones.csv
- modelo_roe_predicciones.parquet
- prophet_forecast.csv
- prophet_forecast.parquet
- prophet_metricas_ajuste.csv
- prophet_metricas_ajuste.parquet
Modelos serializados en: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\04_modelos
- modelo_clusters_kmeans.pkl
- modelo_roe_random_forest.pkl
